In [ ]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path('.')            
OUT_DIR = Path('csv_output')
OUT_DIR.mkdir(exist_ok=True)

LABEL_MAP = {0: 'human', 1: 'bot'}

NUMERIC_COLS = ['followers_count', 'friends_count', 'listed_count',
                'favourites_count', 'statuses_count', 'utc_offset']
BOOL_COLS = ['protected', 'geo_enabled', 'verified', 'contributors_enabled',
             'is_translator', 'is_translation_enabled', 'profile_background_tile',
             'profile_use_background_image', 'has_extended_profile',
             'default_profile', 'default_profile_image']
DROP_COLS = ['id', 'id_str', 'entities', 'profile_location']


def clean_value(v):
    if v is None:
        return None
    v = str(v).strip()
    return None if v in ('None', '') else v


def clean_profile(profile):
    p = {k: clean_value(v) for k, v in profile.items() if k not in DROP_COLS}
    for c in BOOL_COLS:
        if c in p and p[c] is not None:
            p[c] = (p[c] == 'True')
    return p


def convert(split):
    with open(DATA_DIR / f'{split}.json', encoding='utf-8') as f:
        data = json.load(f)

    users, tweets = [], []
    for rec in data:
        uid = str(rec['ID']).strip()
        account_type = LABEL_MAP[int(rec['label'])]

        # one row per account
        row = {'ID': uid, 'account_type': account_type}
        row.update(clean_profile(rec['profile'] or {}))
        domains = rec.get('domain') or []
        row['domain'] = '|'.join(domains)
        for d in ['Politics', 'Business', 'Entertainment', 'Sports']:
            row[f'domain_{d.lower()}'] = int(d in domains)
        tw = rec.get('tweet') or []
        row['n_tweets'] = len(tw)
        users.append(row)

        # one row per tweet
        for i, t in enumerate(tw):
            tweets.append({'ID': uid, 'tweet_idx': i, 'text': t, 'account_type': account_type})

    users_df = pd.DataFrame(users)
    for c in NUMERIC_COLS:
        users_df[c] = pd.to_numeric(users_df[c], errors='coerce')
    users_df['created_at'] = pd.to_datetime(users_df['created_at'],
                                            format='%a %b %d %H:%M:%S %z %Y',
                                            errors='coerce')
    tweets_df = pd.DataFrame(tweets, columns=['ID', 'tweet_idx', 'text', 'account_type'])

    users_df.to_csv(OUT_DIR / f'{split}_users.csv', index=False)
    tweets_df.to_csv(OUT_DIR / f'{split}_tweets.csv', index=False)
    print(split, users_df.shape, tweets_df.shape,
          users_df['account_type'].value_counts().to_dict())
    return users_df, tweets_df


train_users, train_tweets = convert('train')
test_users, test_tweets = convert('test')

train (8278, 42) (1398410, 4) {'bot': 4646, 'human': 3632}
test (1183, 42) (199853, 4) {'bot': 640, 'human': 543}


In [ ]:
COLS_TO_DROP = [
    'utc_offset', 'time_zone', 'lang',
    'protected', 'contributors_enabled', 'is_translator',
    'profile_background_color', 'profile_link_color', 'profile_sidebar_border_color',
    'profile_sidebar_fill_color', 'profile_text_color',
    'profile_background_image_url', 'profile_background_image_url_https',
    'profile_image_url', 'profile_image_url_https',
]

train_users = train_users.drop(columns=COLS_TO_DROP, errors='ignore')
test_users = test_users.drop(columns=COLS_TO_DROP, errors='ignore')

train_users.to_csv(OUT_DIR / 'train_users_cleaned.csv', index=False)
test_users.to_csv(OUT_DIR / 'test_users_cleaned.csv', index=False)

print(train_users.shape, test_users.shape)   

(8278, 27) (1183, 27)
